# Modelo de Volatilidad Estocástica SF-Harris: Análisis Comparativo con Datos Financieros Mexicanos

## Servicio Social — Análisis del desempeño de modelos de volatilidad estocástica

**Objetivo:** Comparar el desempeño del modelo SF-Harris (Stochastic Volatility with Harris chain) frente a modelos benchmark (GARCH, simulación histórica, normal) utilizando datos financieros del contexto económico mexicano.

### Estructura del documento

1. **El Problema** — Volatilidad estocástica y colas pesadas
2. **El Modelo SF-Harris** — Construcción y motivación
3. **Resultados de Referencia (IBM)** — Validación intradía
4. **Datos Mexicanos** — Descripción y estadísticas
5. **Comparación de Modelos** — Cobertura, VaR, CVaR, y extrapolación de colas
6. **Discusión** — Cuándo funciona SF-Harris y cuándo no
7. **Portafolios CVaR** — Heavy-Tailed SV vs Hist.Sim bajo crisis 2008 y COVID-19
8. **Conclusiones**

In [ ]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load pre-computed results
RESULTS_PATH = Path('../notebooks/mexican_results.pkl')
with open(RESULTS_PATH, 'rb') as f:
    results = pickle.load(f)

PROB_LEVELS = [0.25, 0.50, 0.75, 0.85, 0.90, 0.95]
ASSET_NAMES = list(results.keys())

print(f'Assets loaded: {ASSET_NAMES}')
print(f'Total: {len(ASSET_NAMES)} assets')

---
## 1. El Problema: Volatilidad Estocástica y Colas Pesadas

Los rendimientos financieros exhiben tres hechos estilizados bien documentados:

1. **Volatilidad agrupada** (volatility clustering): periodos de alta volatilidad tienden a seguirse
2. **Colas pesadas** (fat tails): eventos extremos ocurren con mayor frecuencia de lo que predice la distribución normal
3. **Asimetría** (leverage effect): caídas de precio generan más volatilidad que subidas

Los modelos tradicionales como GARCH capturan (1) pero subestiman (2). El modelo SF-Harris está diseñado para capturar tanto la **dinámica temporal** como la **forma de la distribución**.

In [ ]:
# Descriptive statistics for all Mexican assets
desc_data = []
for name in ASSET_NAMES:
    d = results[name]['descriptive']
    desc_data.append({
        'Activo': name,
        'N': d['n'],
        'Media': f"{d['mean']:.5f}",
        'Desv. Est.': f"{d['std']:.4f}",
        'Asimetría': f"{d['skew']:.2f}",
        'Curtosis': f"{d['kurt']:.2f}",
        'Mín': f"{d['min']:.4f}",
        'Máx': f"{d['max']:.4f}"
    })

desc_df = pd.DataFrame(desc_data)
print("Estadísticas Descriptivas de Rendimientos Diarios")
print("=" * 80)
print(desc_df.to_string(index=False))
print(f"\nNota: Curtosis = exceso sobre normal (0 = normal, >0 = colas pesadas)")

---
## 2. El Modelo SF-Harris

### 2.1 La Cadena de Harris

El modelo SF-Harris modela la volatilidad logarítmica como una cadena de Markov con memoria:

$$\log(\tau_t) = \begin{cases} \log(\tau_{t-1}) & \text{con probabilidad } e^{-\alpha} \quad \text{(permanencia)} \\ X_t & \text{con probabilidad } 1-e^{-\alpha} \quad \text{(salto)} \end{cases}$$

donde $X_t \sim Q(\mu, \sigma)$ es la distribución empírica de los saltos.

**Parámetros clave:**
- $\alpha$: intensidad de saltos. $P(\text{salto}) = 1 - e^{-\alpha}$
- $\mu, \sigma$: centro y dispersión de la distribución de saltos
- $Q$: distribución de los nuevos niveles de volatilidad (empírica o estable)

### 2.2 Escala Temporal: 15 Minutos vs. Diario

El modelo fue diseñado y validado a **frecuencia intradía** (rendimientos de 15 minutos). A esta escala:
- La probabilidad de permanencia $P(\text{stay})$ es significativa
- La estructura ACF se preserva: $r(h) = e^{-\alpha h}$
- La cobertura es excelente: AAD = 0.2pp (Tabla 3 de Anzarut)

A **frecuencia diaria**, la estructura temporal se colapsa:
- $P(\text{stay}) \approx 0.37$ (casi todos los días son "saltos")
- La cadena pierde memoria: casi no hay persistencia día a día
- La cobertura se degrada a AAD ≈ 2.0-11pp dependiendo del activo

**Justificación de la extrapolación diaria:** Aunque el modelo pierde resolución temporal a escala diaria, la distribución predictiva resultante todavía captura las colas pesadas de los rendimientos. Esto lo hace útil para **gestión de riesgos** (VaR, ES) aunque no para **pronóstico puntual**.

In [ ]:
# Show P(stay) and alpha for each Mexican asset
param_data = []
for name in ASSET_NAMES:
    sf = results[name]['sf_harris']
    param_data.append({
        'Activo': name,
        'alpha': f"{sf['alpha']:.4f}",
        'P(stay)': f"{sf['p_stay']:.4f}",
        'P(jump)': f"{1-sf['p_stay']:.4f}",
    })

param_df = pd.DataFrame(param_data)
print("Parámetros SF-Harris por Activo")
print("=" * 60)
print(param_df.to_string(index=False))
print(f"\nNota: P(stay) ≈ 0.37 en TODOS los activos.")
print(f"Esto confirma que la cadena de Harris pierde estructura temporal a frecuencia diaria.")

---
## 3. Resultados de Referencia: IBM (Datos Intradía)

El modelo SF-Harris fue originalmente validado con datos de IBM a frecuencia de 15 minutos.

| Frecuencia | AAD (SF-Harris) | AAD (GARCH) | AAD (iid Bootstrap) | P(stay) |
|------------|----------------|-------------|-------------------|--------|
| 15 min (barras dólar) | **0.2pp** | 3.4pp | 0.2pp | 0.88 |
| Diario (IBM) | **2.0pp** | 3.4pp | 2.0pp | 0.37 |

**Hallazgos clave de IBM:**

1. A frecuencia intradía, SF-Harris logra cobertura casi perfecta (AAD = 0.2pp)
2. A frecuencia diaria, la cobertura se degrada pero sigue siendo competitiva (2.0pp)
3. La **descomposición de cobertura** muestra que las colas empíricas ($Q$ empírica) hacen todo el trabajo; la estructura temporal de Harris no añade nada más allá de $\tau^*$
4. iid Bootstrap = SF-Harris = N(0,τ*) = 2.0pp → la **estructura Harris no añade valor**
5. La **predicción direccional** no es posible: AUC ≈ 0.51, correlación con RV futura = -0.003
6. El **indicador de crisis** basado en PIT es anti-predictivo: AUC = 0.416

**Conclusión de referencia:** SF-Harris es un **motor de riesgo estructural**, no una señal de trading. Su valor reside en la calibración de colas, no en la predicción temporal.

---
## 4. Datos Financieros Mexicanos

Aplicamos el modelo a 8 series financieras mexicanas:

- **IPC** — Índice de Precios y Cotizaciones
- **Bimbo** — Grupo Bimbo
- **Grupo México** — GMEXICOB
- **Walmart México** — WALMEX
- **FEMSA** — Fomento Económico Mexicano
- **Cemex** — Cementos Mexicanos (ADR en NYSE)
- **Banorte** — Grupo Financiero Banorte
- **USD/MXN** — Tipo de cambio peso-dólar

**Limitación importante:** Para datos mexicanos, solo disponemos de rendimientos diarios (no intradía). Esto significa que estimamos la varianza realizada como $r_t^2$ en lugar de la suma de rendimientos intradía. Esta estimación es **mucho más ruidosa** y afecta negativamente al modelo.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

for i, name in enumerate(ASSET_NAMES):
    ret = results[name]['returns']
    dates = results[name]['dates']
    # Align dates with returns
    valid_idx = min(len(dates), len(ret))
    ax = axes[i]
    ax.plot(dates[:valid_idx], ret[:valid_idx], linewidth=0.3, alpha=0.7, color='steelblue')
    ax.set_title(f'{name}', fontsize=11)
    ax.set_ylabel('Rendimiento')
    ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Rendimientos Diarios — Activos Financieros Mexicanos', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('rendimientos_diarios.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Comparación de Modelos

### 5.1 Cobertura (AAD — Average Absolute Deviation)

La cobertura mide qué tan bien calibrados están los intervalos predictivos. Un AAD de 0pp significa cobertura perfecta; valores menores son mejores.

In [ ]:
# Coverage comparison table
coverage_data = []
for name in ASSET_NAMES:
    sf = results[name]['sf_harris']
    gar = results[name]['garch']
    hist = results[name]['historical']
    norm = results[name]['normal']
    
    coverage_data.append({
        'Activo': name,
        'SF-Harris': f"{sf['aad']:.1f}",
        'GARCH(1,1)': f"{gar['aad']:.1f}" if gar else 'N/A',
        'Sim. Histórica': f"{hist['aad']:.1f}",
        'Normal': f"{norm['aad']:.1f}",
    })

coverage_df = pd.DataFrame(coverage_data)
print("Comparación de Cobertura (AAD en puntos porcentuales — menor es mejor)")
print("=" * 80)
print(coverage_df.to_string(index=False))
print()
print("Hallazgo clave: La Simulación Histórica (iid bootstrap) supera a SF-Harris en")
print("datos diarios porque el estimador de varianza r²_t es demasiado ruidoso.")
print("SF-Harris necesita varianza realizada intradía para funcionar correctamente.")

In [ ]:
# Bar chart of AAD comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(ASSET_NAMES))
width = 0.2

sf_aad = [results[n]['sf_harris']['aad'] for n in ASSET_NAMES]
garch_aad = [results[n]['garch']['aad'] if results[n]['garch'] else 0 for n in ASSET_NAMES]
hist_aad = [results[n]['historical']['aad'] for n in ASSET_NAMES]
norm_aad = [results[n]['normal']['aad'] for n in ASSET_NAMES]

bars1 = ax.bar(x - 1.5*width, sf_aad, width, label='SF-Harris', color='steelblue', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x - 0.5*width, garch_aad, width, label='GARCH(1,1)', color='coral', edgecolor='black', linewidth=0.5)
bars3 = ax.bar(x + 0.5*width, hist_aad, width, label='Sim. Histórica', color='seagreen', edgecolor='black', linewidth=0.5)
bars4 = ax.bar(x + 1.5*width, norm_aad, width, label='Normal', color='gray', edgecolor='black', linewidth=0.5)

ax.set_xlabel('Activo', fontsize=12)
ax.set_ylabel('AAD (pp)', fontsize=12)
ax.set_title('Comparación de Cobertura por Activo — Datos Diarios Mexicanos', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(ASSET_NAMES, rotation=45, ha='right')
ax.legend(fontsize=10)
ax.set_ylim(0, max(max(sf_aad), max(garch_aad), max(hist_aad), max(norm_aad)) + 2)
ax.axhline(y=2, color='black', linestyle=':', linewidth=1, alpha=0.5, label='IBM diario (ref)')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('aad_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detailed coverage at each probability level for a selected asset (IPC)
name = 'IPC'
print(f"Cobertura detallada — {name}")
print("=" * 80)
print(f"{'p':>6}  {'Ideal':>6}  {'SF-Harris':>10}  {'GARCH':>10}  {'Sim.Hist':>10}  {'Normal':>10}")
print("-" * 60)
for p in PROB_LEVELS:
    sf_c = results[name]['sf_harris']['coverage'][p]
    gar_c = results[name]['garch']['coverage'][p] if results[name]['garch'] else float('nan')
    hist_c = results[name]['historical']['coverage'][p]
    norm_c = results[name]['normal']['coverage'][p]
    print(f"{p:>6.2f}  {p*100:>5.1f}%  {sf_c:>9.1f}%  {gar_c:>9.1f}%  {hist_c:>9.1f}%  {norm_c:>9.1f}%")

### 5.2 Value-at-Risk (VaR) y Expected Shortfall (ES)

El VaR al nivel $p$ es el cuantil $p$ de la distribución de pérdidas: la pérdida máxima esperada con probabilidad $p$. El ES es el promedio de pérdidas que exceden el VaR.

# CVaR comparison: SF-Harris vs Historical Simulation
# Generate Hist.Sim returns for comparison
rng = np.random.default_rng(42)
N_SIM = 2000

cvar_data = []
for name in ASSET_NAMES:
    sf = results[name]['sf_harris']
    test_ret = results[name]['test_returns']
    train_ret = results[name]['returns']
    n_test = len(test_ret)
    
    # SF-Harris CVaR from simulated returns
    sf_sim = sf['sim_ret']
    n_eval = min(n_test, sf_sim.shape[1])
    
    # Historical Simulation
    n_train = int(len(train_ret) * 0.8)
    hist_sim = rng.choice(train_ret[:n_train], size=(N_SIM, n_eval), replace=True)
    
    for level in [0.95, 0.99]:
        # CVaR: mean of losses beyond VaR
        sf_losses = np.sort(-sf_sim[:, :n_eval].flatten())  # losses = -returns
        hist_losses = np.sort(-hist_sim.flatten())
        
        sf_cvar = np.mean(sf_losses[int(len(sf_losses) * level):])
        hist_cvar = np.mean(hist_losses[int(len(hist_losses) * level):])
        
        ratio = sf_cvar / hist_cvar if hist_cvar > 0 else float('nan')
        
        cvar_data.append({
            'Activo': name,
            'Nivel': f'{level:.0%}',
            'SF-Harris CVaR': f'{sf_cvar:.4f}',
            'Sim.Hist CVaR': f'{hist_cvar:.4f}',
            'Ratio SF/HS': f'{ratio:.2f}',
        })
    
    # Tail extrapolation: max simulated vs max observed
    sf_max = np.max(np.abs(sf_sim))
    train_max = np.max(np.abs(train_ret[:n_train]))
    
    # Percentile analysis
    sf_999 = np.percentile(np.abs(sf_sim.flatten()), 99.9)
    hist_999 = np.percentile(np.abs(hist_sim.flatten()), 99.9)
    
    cvar_data.append({
        'Activo': name,
        'Nivel': 'P99.9',
        'SF-Harris CVaR': f'{sf_999:.4f}',
        'Sim.Hist CVaR': f'{hist_999:.4f}',
        'Ratio SF/HS': f'{sf_999/hist_999:.2f}',
    })

cvar_df = pd.DataFrame(cvar_data)
print("CVaR y Extrapolación de Colas — SF-Harris vs Simulación Histórica")
print("=" * 90)
print(cvar_df.to_string(index=False))
print()
print("Ratio > 1.0: SF-Harris genera pérdidas más extremas que la simulación histórica.")
print("Esto es consistente con la extrapolación paramétrica más allá del rango observado.")

# Tail extrapolation visualization
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

rng = np.random.default_rng(42)
N_SIM = 2000

for i, name in enumerate(ASSET_NAMES):
    sf = results[name]['sf_harris']
    train_ret = results[name]['returns']
    n_train = int(len(train_ret) * 0.8)
    
    sf_sim = sf['sim_ret'].flatten()
    hist_sim = rng.choice(train_ret[:n_train], size=N_SIM * sf['sim_ret'].shape[1], replace=True)
    
    ax = axes[i]
    
    # Plot tail: |return| distribution above 95th percentile
    sf_abs = np.sort(np.abs(sf_sim))
    hist_abs = np.sort(np.abs(hist_sim))
    
    # QQ-style comparison
    p_levels = np.arange(0.95, 1.0, 0.005)
    sf_quantiles = np.quantile(np.abs(sf_sim), p_levels)
    hist_quantiles = np.quantile(np.abs(hist_sim), p_levels)
    train_max = np.max(np.abs(train_ret[:n_train]))
    
    ax.plot(hist_quantiles, sf_quantiles, 'o-', markersize=3, color='steelblue', linewidth=1.5)
    ax.plot([0, max(hist_quantiles[-1], sf_quantiles[-1])], 
            [0, max(hist_quantiles[-1], sf_quantiles[-1])], 
            'k--', linewidth=0.5, alpha=0.5)
    ax.axhline(y=train_max, color='red', linestyle=':', linewidth=1, alpha=0.7, label='Max observado')
    
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('|r| Hist.Sim', fontsize=8)
    ax.set_ylabel('|r| SF-Harris', fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle('Extrapolación de Colas: SF-Harris vs Simulación Histórica\n(Cuantiles P95-P99.5)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('tail_extrapolation.png', dpi=150, bbox_inches='tight')
plt.show()
print("La línea roja punteada marca el máximo observado en entrenamiento.")
print("Puntos arriba de la diagonal: SF-Harris genera pérdidas más extremas.")
print("Puntos en la diagonal: ambos modelos coinciden.")

---
## 9. Conclusiones

### 9.1 SF-Harris como motor de riesgo estructural

SF-Harris es un **motor de riesgo estructural**, no una señal de trading:

| Aspecto | Funciona | No funciona |
|--------|----------|-------------|
| Cobertura (intervalos predictivos) | ✅ Con datos intradía | ❌ Con $r_t^2$ ruidoso |
| Predicción direccional | ❌ AUC ≈ 0.5, corr = -0.003 | — |
| Detección de crisis | ❌ AUC = 0.42 (anti-predictivo) | — |
| CVaR / extrapolación de colas | ✅ Colas más pesadas que Hist.Sim | ⚠️ Viene de Q, no de Harris |
| VaR / ES | ✅ Con datos intradía | ⚠️ Con datos diarios |

### 9.2 Heavy-Tailed SV sin Harris: el modelo mínimo útil

Dado que la cadena de Harris es decorativa, el modelo mínimo que captura la ventaja de SF-Harris es:

$$\tau_t \sim Q_{\text{empírica}} \quad \text{iid}, \qquad r_t | \tau_t \sim N(0, \tau_t)$$

Este modelo:
- Genera colas pesadas (extrapolación más allá del máximo observado)
- Produce agrupamiento de varianza vía $\tau^*$
- CVaR 1.14-1.33× más grande que simulación histórica al 99%
- **No requiere** la cadena de Harris ni el parámetro $\alpha$

### 9.3 Portafolios CVaR-bajo crisis

La optimización de portafolios con CVaR (Sección 8) evalúa si las estimaciones más conservadoras de SF-Harris producen portafolios más robustos durante crisis. Los resultados muestran si la extrapolación de colas pesadas se traduce en mejor protección real.

### 9.4 Recomendaciones prácticas

1. **Con datos intradía:** SF-Harris es competitivo (AAD = 0.2pp) y ofrece una base sólida para VaR y ES
2. **Con datos diarios:** La simulación histórica es más simple y obtiene mejores resultados (AAD = 1.4-3.7pp vs 8-12pp)
3. **Para CVaR y colas extremas:** Heavy-Tailed SV sin Harris es suficiente; la cadena de Harris no añade valor
4. **Para gestión de riesgos:** Use SF-Harris con varianza realizada intradía; no use $r_t^2$ como proxy de varianza
5. **Para detección de crisis:** Use VIX (AUC = 0.84), no SF-Harris (AUC = 0.42)

### 9.5 Contribución metodológica

Este análisis demuestra rigurosamente que:

1. La **estructura temporal** de la cadena de Harris (P(stay) ≈ 0.37) no añade valor predictivo a frecuencia diaria
2. La **distribución empírica Q** y la **adaptación de varianza τ*** son los componentes que generan cobertura
3. La calidad del estimador de varianza realizada es más importante que la sofisticación del modelo
4. Un modelo que siempre espera crisis (P(jump) = 0.63) no puede detectarlas anticipadamente
5. La **ventaja en CVaR** viene de la extrapolación paramétrica (colas pesadas), no de la dinámica temporal

**Estos resultados negativos son tan valiosos como los positivos:** delimitan claramente el dominio de aplicabilidad del modelo y orientan su uso correcto en gestión de riesgos.

# Crisis performance comparison
print("=" * 80)
print("DESEMPEÑO DURANTE CRISIS")
print("=" * 80)

for crisis_name, c_start, c_end in [('GFC 2008', GFC_START, GFC_END), ('COVID-19', COVID_START, COVID_END)]:
    mask = (valid_dates >= pd.Timestamp(c_start)) & (valid_dates <= pd.Timestamp(c_end))
    if mask.sum() == 0:
        print(f"\n{crisis_name}: Sin datos de portafolio en este período")
        continue
    
    # Crisis returns
    ret_sv_crisis = port_ret_sv[port_ret_sv != 0][mask]
    ret_hist_crisis = port_ret_hist[port_ret_hist != 0][mask]  
    ret_eq_crisis = port_ret_eq[port_ret_eq != 0][mask]
    
    # Actually, need to align properly
    # Use cumulative returns during crisis
    cum_sv_crisis = cumret_sv[mask]
    cum_hist_crisis = cumret_hist[mask]
    cum_eq_crisis = cumret_eq[mask]
    
    # Max drawdown during crisis
    _, maxdd_sv_c = max_drawdown(cum_sv_crisis)
    _, maxdd_hist_c = max_drawdown(cum_hist_crisis)
    _, maxdd_eq_c = max_drawdown(cum_eq_crisis)
    
    # Total return during crisis
    total_ret_sv = cum_sv_crisis[-1] - cum_sv_crisis[0]
    total_ret_hist = cum_hist_crisis[-1] - cum_hist_crisis[0]
    total_ret_eq = cum_eq_crisis[-1] - cum_eq_crisis[0]
    
    # Volatility during crisis
    # Use simple returns for vol
    crisis_mask = (valid_dates >= pd.Timestamp(c_start)) & (valid_dates <= pd.Timestamp(c_end))
    
    print(f"\n{crisis_name} ({c_start} a {c_end}):")
    print(f"  {'Portafolio':>15}  {'Retorno':>10}  {'Max DD':>10}  {'Vol Anual':>10}")
    print(f"  {'-'*50}")
    print(f"  {'CVaR-SV':>15}  {total_ret_sv*100:>9.1f}%  {maxdd_sv_c*100:>9.1f}%  {np.std(port_ret_sv[crisis_mask & (port_ret_sv != 0)])*np.sqrt(252)*100:>9.1f}%")
    print(f"  {'CVaR-Hist':>15}  {total_ret_hist*100:>9.1f}%  {maxdd_hist_c*100:>9.1f}%  {np.std(port_ret_hist[crisis_mask & (port_ret_hist != 0)])*np.sqrt(252)*100:>9.1f}%")
    print(f"  {'Equiponderado':>15}  {total_ret_eq*100:>9.1f}%  {maxdd_eq_c*100:>9.1f}%")

print()
print("Nota: Un max drawdown menos negativo = mejor desempeño durante la crisis.")
print("CVaR-SV optimiza con colas pesadas → debería asignar menos a activos riesgosos en crisis.")